# 06d · GSE65391 · RNA_array · patient-group profiles (preprocessing for 06e–06g)

Reads the series matrix (probe level), `metadata.rds` (notebook 01), `projected_scores.rds` (notebook 05b)
and Table S5 of Banchereau et al. Writes `pg_profiles.rds`, `pg_profile_797.csv` and
`pg_profile_modules16.csv` in `data/run_artifacts/GSE65391/`. No clustering is done here.

**Claim B6** (Banchereau R et al. Personalized immunomonitoring uncovers molecular networks that stratify
lupus patients. *Cell* 2016;165:551–565, doi:10.1016/j.cell.2016.03.008): 80 children with five or more
visits fall into seven patient groups (PG1–7), each defined by which of five immune networks follow the
child's disease activity. The paper does not publish which child is in which group (checked: Tables S1–S5
and the methods PDF; GEO has no such field). Notebooks 06d–06g therefore rebuild the grouping.

**What the paper publishes and this notebook uses.**
- **Table S5** (`NIHMS775280-supplement-12.xlsx`, downloaded by hand from PMC5426482): the 797 array probes
  used for stratification, each assigned to one of five networks: ER (erythropoiesis), IFN (interferon),
  ML (myeloid lineage / neutrophils), PC (plasma cells), LL (lymphoid lineage).
- **The patient profile** (supplementary methods, "Classification of independent SLE patients"): for each
  child, "the matrix of SLEDAI correlations with the 797 transcripts ... collapsed by immune network".

**Profiles built here, one row per child:**
1. `r797`: Pearson r between SLEDAI and each of the 797 probes, across that child's visits.
2. `net5`: the mean of `r797` within each of the five networks (the paper's collapsed profile).
3. `mod16`: Pearson r between SLEDAI and each of our 16 module scores (notebook 05b) across the child's
   visits, for the side-by-side comparison with our own modules.

**Children.** Children with SLE and five or more samples with a recorded SLEDAI. A correlation needs SLEDAI
to vary, so children whose SLEDAI is the same at every visit are excluded and counted.

**Check against the paper.** Table S2 gives the 80 children's totals (race, sex, mean visits). If the
children selected here match those totals, the same children were very likely used.

In [1]:
source("../src/paths.R")
suppressMessages({library(GEOquery); library(Biobase); library(readxl)})
meta <- readRDS(art("GSE65391", "metadata.rds"))
f5 <- raw("GSE65391", "NIHMS775280-supplement-12.xlsx")
stopifnot(file.exists(f5))
title <- names(read_excel(f5, n_max = 0))[1]; title
s5 <- as.data.frame(read_excel(f5, skip = 2))
X  <- exprs(suppressMessages(getGEO(filename = raw("GSE65391", "GSE65391_series_matrix.txt.gz"), getGPL = FALSE)))
c(probes_listed = nrow(s5), unique = length(unique(s5$ProbeID)), on_array = sum(s5$ProbeID %in% rownames(X)))
table(network = s5$Network)

[1] "Table S5 - 797 transcripts for SLE patient stratification - Related to Figure 7."

probes_listed        unique      on_array 
          797           797           797

network
 ER IFN  LL  ML  PC 
149  40 436 163   9 

**Result.** Table S5 lists 797 probes; the counts by network are those given in the paper's text
(ER 149, IFN 40, ML 163, PC 9, LL 436).

In [2]:
sle <- meta[meta$disease == "SLE" & !is.na(meta$sledai), ]
n_visits <- table(sle$subject)
five <- names(n_visits)[n_visits >= 5]
varies <- tapply(sle$sledai, sle$subject, function(v) length(unique(v)) > 1)
kids <- five[varies[five]]
c(children_with_SLE = length(n_visits), five_or_more_visits = length(five),
  of_which_SLEDAI_constant = sum(!varies[five]), used = length(kids))

children_with_SLE      five_or_more_visits of_which_SLEDAI_constant 
                     158                       84                        1 
                    used 
                      83

**Result.** 84 of the 158 children have five or more visits with a SLEDAI. One has the same SLEDAI at
every visit and is excluded, leaving 83; the paper stratified 80.

**Check against Table S2** (totals for the paper's 80 children): race AA 21, C 11, H 48; female 72,
male 8; mean number of visits 9.29.

In [3]:
kid <- data.frame(subject = kids,
                  race = sle$race[match(kids, sle$subject)], sex = sle$sex[match(kids, sle$subject)],
                  visits = as.integer(n_visits[kids]),
                  mean_sledai = round(tapply(sle$sledai, sle$subject, mean)[kids], 1), row.names = kids)
list(race = table(kid$race), sex = table(kid$sex), mean_visits = round(mean(kid$visits), 2))

$race

AA  C  H 
21 11 51 

$sex

 F  M 
74  9 

$mean_visits
[1] 9.16


**Result.** African-American (21) and Caucasian (11) match Table S2 exactly. There are 51 Hispanic
children against the paper's 48, and 74 girls and 9 boys against 72 and 8, so the paper's 80 children
appear to be these 83 less three Hispanic children (two girls, one boy). Mean visits: 9.16 against 9.29.
GEO does not say which three the paper left out, so all 83 are kept.

In [4]:
r_with_sledai <- function(values, k) {       # values: features x samples; Pearson r with SLEDAI over child k's visits
  s <- rownames(sle)[sle$subject == k]
  v <- values[, s, drop = FALSE]
  r <- suppressWarnings(cor(t(v), sle[s, "sledai"])[, 1])
  r[apply(v, 1, sd) == 0] <- NA               # a feature that does not vary within the child has no r
  r
}
r797 <- t(sapply(kids, function(k) r_with_sledai(X[s5$ProbeID, ], k)))
net5 <- sapply(split(s5$ProbeID, s5$Network), function(p) rowMeans(r797[, p, drop = FALSE], na.rm = TRUE))
net5 <- net5[, c("ER", "IFN", "ML", "PC", "LL")]
ps <- readRDS(art("GSE65391", "projected_scores.rds"))
mod16 <- t(sapply(kids, function(k) r_with_sledai(t(ps$scores), k)))
c(children = nrow(r797), probes = ncol(r797), missing_r = sum(is.na(r797)), modules = ncol(mod16))
round(apply(net5, 2, summary), 2)

children    probes missing_r   modules 
       83       797       520        16

,ER,IFN,ML,PC,LL
Min.,-0.62,-0.67,-0.62,-0.71,-0.66
1st Qu.,-0.24,-0.13,-0.10,-0.17,-0.23
Median,-0.03,0.10,0.06,0.09,-0.02
Mean,-0.03,0.10,0.04,0.10,-0.02
3rd Qu.,0.13,0.34,0.23,0.34,0.19
Max.,0.70,0.81,0.50,0.85,0.77


**Result.** 83 children × 797 probes. 520 of the 66,151 correlations are missing, where a probe does not
vary within a child. Averaged by network, the children's SLEDAI correlations run from about −0.7 to
+0.85; the interferon and plasma-cell networks have the highest medians (0.10 and 0.09).

In [5]:
saveRDS(list(children = kid, r797 = r797, net5 = net5, mod16 = mod16, s5 = s5), art("GSE65391", "pg_profiles.rds"))
write.csv(r797, art("GSE65391", "pg_profile_797.csv"))
write.csv(mod16, art("GSE65391", "pg_profile_modules16.csv"))